## GO term enrichment analysis using topGO

In [1]:
library(dplyr)
library(tidyr)
library(stringr)
library(topGO)
library(tibble)
library(ggplot2)
library(cowplot)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: graph


Attaching package: ‘graph’


The following objec

In [2]:
run_GO_enrichment <- function(uniprot_file, genes_file) {
  # 1. Load UniProt annotations
  df <- read.delim(uniprot_file, header = TRUE, sep = "\t", stringsAsFactors = FALSE)

  # 2. Filter genes with GO annotations and valid names
  df <- df %>%
    filter(Gene.Ontology.IDs != "", Gene.Names != "") %>%
    dplyr::select(Gene.Names, Gene.Ontology.IDs)

  # 3. Split multiple GOs into separate rows
  df_split <- df %>%
    mutate(GO = strsplit(Gene.Ontology.IDs, "; ")) %>%
    unnest(GO)

  # 4. Extract only GO IDs
  df_split$GO <- str_extract(df_split$GO, "GO:\\d+")

  # 5. Create gene2GO mapping as a list (without temporary file)
  gene2GO_list <- df_split %>%
    group_by(Gene.Names) %>%
    summarise(GOterms = list(unique(GO)), .groups = "drop") %>%
    tibble::deframe()

  geneNames <- names(gene2GO_list)

  # 6. Load the gene list for the category
  cat_genes <- readLines(genes_file)
  geneList <- factor(as.integer(geneNames %in% cat_genes))
  names(geneList) <- geneNames

  # 7. Create a topGO object directly from the list
  GOdata <- new("topGOdata",
                ontology = "BP",
                allGenes = geneList,
                annot = annFUN.gene2GO,
                gene2GO = gene2GO_list)

  # 8. Run GO test (elim + fisher)
  result_elim <- runTest(GOdata, algorithm = "elim", statistic = "fisher", cutOff = 0.001)

  elim_BP <- GenTable(GOdata,
                      elimFisher = result_elim,
                      orderBy = "elimFisher",
                      topNodes = 20)

  # 9. Format result
  enrich_gos <- dplyr::select(elim_BP, `GO.ID`, Term, Annotated, Significant, elimFisher)
  colnames(enrich_gos) <- c("GO", "Term", "Annotated","Significant", "PValue")
  enrich_gos$PValue <- as.numeric(enrich_gos$PValue)
  enrich_gos$Enrichment <- round(-log10(enrich_gos$PValue), 2)
  enrich_gos$Gene_frequency <- (enrich_gos$Significant)/(enrich_gos$Annotated)
  enrich_gos$Gene_frequency <- round(enrich_gos$Gene_frequency, 3)

  return(enrich_gos)
}


In [3]:
# Open files
data_dir <- "./orgs_with_categories"
files <- list.files(data_dir, pattern = "\\.tsv$", full.names = TRUE)

# Carpeta de salida para los archivos de genes
out_dir <- file.path(getwd(), "gene_lists")
if(!dir.exists(out_dir)) dir.create(out_dir)

# Recorrer cada archivo
for(f in files){
  # Leer datos
  data <- read.delim(f, header = TRUE, stringsAsFactors = FALSE)
  
  # Crear una lista con las categorías
  categorias <- list(
    E = "essential",
    I = "important",
    N = "noeffect", #noeffect
    F = "enhance"
  )
}

In [4]:
categorias

$E
[1] "essential"

$I
[1] "important"

$N
[1] "noeffect"

$F
[1] "enhance"

In [5]:
# 1. Directories and files
# - gene_lists live in `gene_dir/` and are named like: "<category>_<organism>.tsv"
# - UniProt annotation files live in `uniprot_dir/` and are named like: "uniprotkb_<organism>.tsv"
# - `run_GO_enrichment(uniprot_file, genes_file)` returns a data.frame with columns:
#   GO, Term, Annotated, Significant, PValue, Enrichment, Gene_frequency-------

gene_dir <- "./gene_lists"     
uniprot_dir <- "./uniprot_gos/"    

# Load files
gene_files_all <- list.files(gene_dir, full.names = TRUE, pattern = "\\.tsv$")
uniprot_files <- list.files(uniprot_dir, full.names = TRUE, pattern = "\\.tsv$")

# Extract organism names from UniProt files
uniprot_orgs <- sub("^.*/uniprotkb_", "", uniprot_files)
uniprot_orgs <- sub("\\.tsv$", "", uniprot_orgs)
uniprot_orgs <- tolower(trimws(uniprot_orgs))


# 2. Process gene files
df <- data.frame(
  filepath = gene_files_all,
  base = basename(gene_files_all),
  stringsAsFactors = FALSE
)

df$category <- tolower(trimws(sub("_.*", "", df$base)))
df$organism <- tolower(trimws(sub("^[^_]+_", "", sub("\\.tsv$", "", df$base))))

# Category order
cat_order <- c("essential", "important", "noeffect", "enhance")

# List to store results by organism
results_list <- list()


# 3. Loop through organisms
organisms <- unique(df$organism)

for (org in organisms) {
  # Gene files for this organism
  org_files <- df[df$organism == org, ]
  
  # Corresponding UniProt file
  idx <- which(uniprot_orgs == org)
  if(length(idx) == 0) {
    warning(paste("No UniProt file found for", org))
    next
  }
  uniprot_match <- uniprot_files[idx]
  
  # Gene files in order
  gene_files <- sapply(cat_order, function(cat) {
    f <- org_files$filepath[org_files$category == cat]
    if(length(f) == 0) return(NA)
    f
  })
  
  # Run run_GO_enrichment
  results <- lapply(gene_files, function(file) {
    run_GO_enrichment(uniprot_match, file)
  })
  
  # Save into list with category names
  results_list[[org]] <- setNames(results, cat_order)
}



Building most specific GOs .....

	( 135 GO terms found. )


Build GO DAG topology ..........

	( 590 GO terms and 1178 relations. )


Annotating nodes ...............

	( 418 genes annotated to the GO terms. )


			 -- Elim Algorithm -- 

		 the algorithm is scoring 128 nontrivial nodes
		 parameters: 
			 test statistic: fisher
			 cutOff: 0.001


	 Level 13:	2 nodes to be scored	(0 eliminated genes)


	 Level 12:	2 nodes to be scored	(0 eliminated genes)


	 Level 11:	7 nodes to be scored	(0 eliminated genes)


	 Level 10:	12 nodes to be scored	(0 eliminated genes)


	 Level 9:	13 nodes to be scored	(9 eliminated genes)


	 Level 8:	11 nodes to be scored	(20 eliminated genes)


	 Level 7:	11 nodes to be scored	(20 eliminated genes)


	 Level 6:	12 nodes to be scored	(38 eliminated genes)


	 Level 5:	22 nodes to be scored	(38 eliminated genes)


	 Level 4:	20 nodes to be scored	(38 eliminated genes)


	 Level 3:	11 nodes to be scored	(38 eliminated genes)


	 Level 2:	4 nodes to be

In [6]:
# 4. Combine all results into one dataframe
all_gos_list <- list()

for (org in names(results_list)) {
  org_results <- results_list[[org]]
  
  org_dfs <- lapply(names(org_results), function(cat_name) {
    df <- org_results[[cat_name]] %>%
      mutate(Category = cat_name, Organism = org)
    return(df)
  })
  
  org_combined <- bind_rows(org_dfs)
  all_gos_list[[org]] <- org_combined
}

all_gos <- bind_rows(all_gos_list)

# Ensure Category is a factor with the desired order
all_gos$Category <- factor(all_gos$Category, levels = cat_order)
all_gos


GO,Term,Annotated,Significant,PValue,Enrichment,Gene_frequency,Category,Organism
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
GO:0000105,histidine biosynthetic process,11,8,3.400e-09,8.47,0.727,essential,bacteroides_thetaiotaomicron
GO:0000162,tryptophan biosynthetic process,9,6,1.100e-06,5.96,0.667,essential,bacteroides_thetaiotaomicron
GO:0006520,amino acid metabolic process,38,19,1.200e-05,4.92,0.500,essential,bacteroides_thetaiotaomicron
GO:1901605,alpha-amino acid metabolic process,19,9,6.000e-03,2.22,0.474,essential,bacteroides_thetaiotaomicron
GO:0046395,carboxylic acid catabolic process,3,2,8.400e-03,2.08,0.667,essential,bacteroides_thetaiotaomicron
GO:1901606,alpha-amino acid catabolic process,3,2,8.400e-03,2.08,0.667,essential,bacteroides_thetaiotaomicron
GO:0016054,organic acid catabolic process,3,2,8.400e-03,2.08,0.667,essential,bacteroides_thetaiotaomicron
GO:0009063,amino acid catabolic process,3,2,8.400e-03,2.08,0.667,essential,bacteroides_thetaiotaomicron
GO:1901566,organonitrogen compound biosynthetic pro...,41,17,8.700e-03,2.06,0.415,essential,bacteroides_thetaiotaomicron


In [7]:
# Filter significant terms
filtered_gos <- all_gos %>%
  filter(Enrichment > 3, Gene_frequency > 0.25, Annotated >= 2)
filtered_gos


GO,Term,Annotated,Significant,PValue,Enrichment,Gene_frequency,Category,Organism
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
GO:0000105,histidine biosynthetic process,11,8,3.4e-09,8.47,0.727,essential,bacteroides_thetaiotaomicron
GO:0000162,tryptophan biosynthetic process,9,6,1.1e-06,5.96,0.667,essential,bacteroides_thetaiotaomicron
GO:0006520,amino acid metabolic process,38,19,1.2e-05,4.92,0.500,essential,bacteroides_thetaiotaomicron
GO:0002098,tRNA wobble uridine modification,3,3,3.4e-04,3.47,1.000,important,bacteroides_thetaiotaomicron
GO:0005975,carbohydrate metabolic process,112,105,1.4e-10,9.85,0.938,noeffect,bacteroides_thetaiotaomicron
GO:0006189,'de novo' IMP biosynthetic process,14,11,1.5e-10,9.82,0.786,essential,caulobacter_crescentus_na1000
GO:0000105,histidine biosynthetic process,12,9,1.6e-08,7.80,0.750,essential,caulobacter_crescentus_na1000
GO:0009073,aromatic amino acid family biosynthetic ...,17,13,1.2e-06,5.92,0.765,essential,caulobacter_crescentus_na1000
GO:0000162,tryptophan biosynthetic process,7,6,1.4e-06,5.85,0.857,essential,caulobacter_crescentus_na1000


In [8]:
library(GO.db)
library(AnnotationDbi)

# --- List of organisms where redundant GO terms will be filtered more strictly ---
orgs_strict <- c(
  "bacteroides_thetaiotaomicron",
  "caulobacter_crescentus_na1000",
  "dinoroseobacter_shibae_dfl_12",
  "echinicola_vietnamensis",
  "sinorhizobium_meliloti_1021",
  "pseudomonas_fluorescens_fw300-n2c3",
  "pseudomonas_putida_kt2440"
)

# --- Function: get parent terms for a given GO ID ---
# Uses GOBPANCESTOR from GO.db to retrieve biological process ancestors
get_parents <- function(go_id) {
  parents <- AnnotationDbi::mget(go_id, GOBPANCESTOR, ifnotfound = NA) 
  unlist(parents, use.names = FALSE)
}

# --- Function: remove GO terms that are parents of other terms in the same set ---
# Keeps only "leaf" terms (i.e., not higher-level ancestors of others in the result)
remove_parent_terms <- function(df) {
  go_list <- df$GO
  is_parent <- sapply(
    go_list,
    function(go) any(go %in% unlist(lapply(go_list, get_parents)), na.rm = TRUE)
  )
  df[!is_parent, ]
}

filtered_strict <- all_gos %>%
  filter(
    Organism %in% orgs_strict,
    Enrichment > 3,
    Gene_frequency > 0.3,
    Annotated >= 2
  ) %>%
  remove_parent_terms()

filtered_light <- all_gos %>%
  filter(
    !(Organism %in% orgs_strict),
    Enrichment > 2,
    Gene_frequency > 0.3,
    Annotated >= 2
  )

# --- 3) Combine both filtered sets ---
filtered_gos <- bind_rows(filtered_light, filtered_strict)
filtered_gos


GO,Term,Annotated,Significant,PValue,Enrichment,Gene_frequency,Category,Organism
<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
GO:0000105,histidine biosynthetic process,13,9,2.7e-12,11.57,0.692,essential,pseudomonas_simiae_wcs417
GO:0000162,tryptophan biosynthetic process,6,5,1.3e-07,6.89,0.833,essential,pseudomonas_simiae_wcs417
GO:0006865,amino acid transport,28,28,1.4e-04,3.85,1.000,noeffect,pseudomonas_simiae_wcs417
GO:0000105,histidine biosynthetic process,12,9,1.0e-12,12.00,0.750,essential,pseudomonas_stutzeri_rch2
GO:0000162,tryptophan biosynthetic process,6,5,1.5e-07,6.82,0.833,essential,pseudomonas_stutzeri_rch2
GO:0032787,monocarboxylic acid metabolic process,22,21,6.6e-03,2.18,0.955,noeffect,pseudomonas_stutzeri_rch2
GO:0042221,response to chemical,22,21,6.6e-03,2.18,0.955,noeffect,pseudomonas_stutzeri_rch2
GO:0050896,response to stimulus,97,80,7.1e-03,2.15,0.825,noeffect,pseudomonas_stutzeri_rch2
GO:0000105,histidine biosynthetic process,11,8,3.4e-09,8.47,0.727,essential,bacteroides_thetaiotaomicron


In [9]:
# --- 1) Strict filtering without removing parent terms ---
filtered_strict_nor <- all_gos %>%
  filter(
    Organism %in% orgs_strict,
    Enrichment > 3,
    Gene_frequency > 0.3,
    Annotated >= 2
  )

# --- 2) Strict filtering with removal of parent terms ---
filtered_strict_remove_parents <- all_gos %>%
  filter(
    Organism %in% orgs_strict,
    Enrichment > 3,
    Gene_frequency > 0.3,
    Annotated >= 2
  ) %>%
  remove_parent_terms()

filtered_strict_remove_parents

# --- 3) Additional filter: exclude the "important" category ---
filter_imp <- filtered_gos %>%
  filter(Category != "important")

,GO,Term,Annotated,Significant,PValue,Enrichment,Gene_frequency,Category,Organism
,<chr>,<chr>,<int>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<chr>
1,GO:0000105,histidine biosynthetic process,11,8,3.4e-09,8.47,0.727,essential,bacteroides_thetaiotaomicron
2,GO:0000162,tryptophan biosynthetic process,9,6,1.1e-06,5.96,0.667,essential,bacteroides_thetaiotaomicron
4,GO:0002098,tRNA wobble uridine modification,3,3,3.4e-04,3.47,1.000,important,bacteroides_thetaiotaomicron
6,GO:0006189,'de novo' IMP biosynthetic process,14,11,1.5e-10,9.82,0.786,essential,caulobacter_crescentus_na1000
7,GO:0000105,histidine biosynthetic process,12,9,1.6e-08,7.80,0.750,essential,caulobacter_crescentus_na1000
9,GO:0000162,tryptophan biosynthetic process,7,6,1.4e-06,5.85,0.857,essential,caulobacter_crescentus_na1000
12,GO:0009423,chorismate biosynthetic process,7,5,5.3e-05,4.28,0.714,essential,caulobacter_crescentus_na1000
13,GO:0009098,leucine biosynthetic process,5,4,1.8e-04,3.74,0.800,essential,caulobacter_crescentus_na1000
14,GO:0009086,methionine biosynthetic process,9,5,2.8e-04,3.55,0.556,essential,caulobacter_crescentus_na1000


In [11]:
sessionInfo()

R version 4.3.3 (2024-02-29)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 22.04.5 LTS

Matrix products: default
BLAS/LAPACK: /apps/bioconda2/envs/r432/lib/libopenblasp-r0.3.29.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8           LC_NUMERIC=C              
 [3] LC_TIME=es_MX.UTF-8        LC_COLLATE=C.UTF-8        
 [5] LC_MONETARY=es_MX.UTF-8    LC_MESSAGES=C.UTF-8       
 [7] LC_PAPER=es_MX.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=es_MX.UTF-8 LC_IDENTIFICATION=C       

time zone: America/Mexico_City
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] cowplot_1.2.0        ggplot2_3.5.2        tibble_3.3.0        
 [4] topGO_2.54.0         SparseM_1.81         GO.db_3.18.0        
 [7] AnnotationDbi_1.64.1 IRanges_2.36.0       S4Vectors_0.40.2    
[10] Bioba

In [12]:
# --- Función base si fun
make_plot <- function(df, title, major_linewidth = 0.3, wrap_width = 50, alpha = 0.7) {
  ggplot(df, aes(x = Category, y = Term,
                 color = Enrichment, size = Gene_frequency)) +
    geom_point() +
    theme_minimal() +
    scale_color_viridis_c(name = '-log10 (pvalue)') +
    scale_x_discrete(
      limits = cat_order,
      labels = c(
        "essential" = "essential",
        "noeffect"  = "mean effect",
        "enhance"   = "enhance"
      )
    ) +
    labs(title = stringr::str_wrap(title, width = wrap_width),
         x = NULL, y = NULL,
         color = "Enrichment",
         size = "Gene frequency") +
    theme(
      plot.background   = element_rect(fill = "white", colour = NA),
      panel.background  = element_rect(fill = "white", colour = NA),
      panel.grid.major  = element_line(color = "grey50", linewidth = major_linewidth),
      panel.grid.minor  = element_line(color = "grey70", linewidth = 0.25),
      axis.line         = element_blank(),
      axis.text.x       = element_text(vjust = 0.9, hjust = 0.5,
                                       colour = "black", angle = 45),
      axis.text.y       = element_text(colour = "black", size = 10),
      axis.ticks        = element_blank(),
      legend.position   = "none",
      plot.title        = element_text(hjust = 0.5, size = 12, face = "bold.italic"),
      plot.margin       = margin(10, 10, 10, 28)
    ) +
    coord_cartesian(clip = "off")
}

# --- Organism keys and display titles ---
orgs <- list(
  "caulobacter_crescentus_na1000"       = "C. crescentus",
  "pseudomonas_fluorescens_fw300-n2c3"  = "P. fluorescens",
  "pseudomonas_putida_kt2440"           = "P. putida",
  "dinoroseobacter_shibae_dfl_12"       = "D. shibae",
  "pseudomonas_simiae_wcs417"           = "P. simiae",
  "pseudomonas_stutzeri_rch2"           = "P. stutzeri",
  "sinorhizobium_meliloti_1021"         = "S. meliloti",
  "echinicola_vietnamensis"             = "E. vietnamensis"
)

# --- Generar plots por organismo
plots <- lapply(names(orgs), function(org) {
  df_org <- filter_imp %>% dplyr::filter(Organism == org)
  make_plot(df_org, orgs[[org]], major_linewidth = 0.3)
})

legend_plot <- ggplot(filter_imp, aes(x = Category, y = Term,
                                      color = Enrichment, size = Gene_frequency)) +
  geom_point() +
  theme_minimal(base_size = 9) +                         
  scale_color_viridis_c(name = "-log10 (pvalue)") +
  scale_size(name = "Gene frequency", range = c(2.0, 5.0)) +
  theme(
    legend.position       = "right",
    legend.direction      = "vertical",
    legend.title          = element_text(size = 10, face = "bold"),
    legend.text           = element_text(size = 9),
    legend.background     = element_rect(fill = "white", colour = NA),  # <- fondo blanco
    legend.box.background = element_rect(fill = "white", colour = NA),
    legend.key            = element_rect(fill = "white", colour = NA)
  )
legend <- cowplot::get_legend(legend_plot)

lots <- Map(function(p, i){
  if (i %in% 1:3) {
    p + theme(plot.margin = margin(10, 3, 5, 3))   # top, right, bottom, left
  } else if (i %in% 4:6) {
    p + theme(plot.margin = margin(5, 3, 5, 3))
  } else {
    p + theme(plot.margin = margin(5, 3, 5, 3))
  }
}, plots, seq_along(plots))


# --- Columnas (sin cambios de estilo en tus plots) ---
col1 <- cowplot::plot_grid(plots[[1]], plots[[2]], plots[[3]],
                           ncol = 1, align = "v", axis = "l")
col2 <- cowplot::plot_grid(plots[[4]], plots[[5]], plots[[6]],
                           ncol = 1, align = "v", axis = "l")
col3 <- cowplot::plot_grid(plots[[7]], plots[[8]],
    cowplot::ggdraw() +
    cowplot::draw_grob(legend,
                       x = -0.5, y = 0.5,           
                    #   width = 0.9, height = 0.9, # padding interno (10%)
                       hjust = 0.5, vjust = 0.5), # ancla centrada
  ncol = 1, align = "v", axis = "l",
  rel_heights = c(1, 1, 1)
)

# --- Figura combinada (mantengo tu alineación vertical) ---
combined <- cowplot::plot_grid(
  col1, col2, col3,
  ncol = 3, align = "v", axis = "l",
  greedy = TRUE
)

ggsave("./Figures/Fig06_dotplot.svg",
       plot = combined,
       device = "svg",  #
       width = 14, height = 11, units = "in",
       dpi = 300,
       bg = "white")#,
       #compression = "lzw")

In [13]:
make_plot <- function(df, title,
                      major_linewidth = 0.3,
                      wrap_width = 50,
                      jitter_w = 0.08, jitter_h = 0.08,        # jitter suave (ensayo 0.06–0.12)
                      alpha_range = c(0.25, 0.9),               # más puntos => más opaco
                      size_range = c(2.2, 7.5)                  # círculos más grandes
) {
  # 1) Contar coincidencias por (Category, Term) para controlar alpha
  df2 <- df %>%
    dplyr::count(Category, Term, name = "overlap_n") %>%
    dplyr::right_join(df, by = c("Category","Term"))

  ggplot(df2, aes(x = Category, y = Term,
                  color = Enrichment,
                  size  = Gene_frequency,
                  alpha = overlap_n)) +                         
    geom_point(position = position_jitter(width = jitter_w,
                                          height = jitter_h,
                                          seed = 1)) +          
    theme_minimal() +
    scale_color_viridis_c(name = '-log10 (pvalue)') +
    scale_size(name = "Gene frequency", range = size_range) +   # controla tamaño global
    scale_alpha(range = alpha_range, guide = "none") +          # sin leyenda de alpha
    scale_x_discrete(
      limits = cat_order,
      labels = c("essential" = "essential",
                 "noeffect"  = "mean effect",
                 "enhance"   = "enhance")
    ) +
    labs(title = stringr::str_wrap(title, width = wrap_width),
         x = NULL, y = NULL,
         color = "Enrichment",
         size = "Gene frequency") +
    theme(
      plot.background   = element_rect(fill = "white", colour = NA),
      panel.background  = element_rect(fill = "white", colour = NA),
      panel.grid.major  = element_line(color = "grey50", linewidth = major_linewidth),
      panel.grid.minor  = element_line(color = "grey70", linewidth = 0.25),
      axis.line         = element_blank(),
      axis.text.x       = element_text(vjust = 0.9, hjust = 0.5,
                                       colour = "black", angle = 45),
      axis.text.y       = element_text(colour = "black", size = 10),
      axis.ticks        = element_blank(),
      legend.position   = "none",
      plot.title        = element_text(hjust = 0.5, size = 12, face = "bold.italic"),
      plot.margin       = margin(10, 20, 10, 16)                # margen izq un poco más compacto
    ) +
    coord_cartesian(clip = "off")
}

# --- Organism keys and display titles ---
orgs <- list(
  "caulobacter_crescentus_na1000"       = "C. crescentus",
  "pseudomonas_fluorescens_fw300-n2c3"  = "P. fluorescens",
  "pseudomonas_putida_kt2440"           = "P. putida",
  "dinoroseobacter_shibae_dfl_12"       = "D. shibae",
  "pseudomonas_simiae_wcs417"           = "P. simiae",
  "pseudomonas_stutzeri_rch2"           = "P. stutzeri",
  "sinorhizobium_meliloti_1021"         = "S. meliloti",
  "echinicola_vietnamensis"             = "E. vietnamensis"
)

# --- Generar plots por organismo (usa 'filter_imp' ya filtrado) ---
plots <- lapply(names(orgs), function(org) {
  df_org <- filter_imp %>% dplyr::filter(Organism == org)
  make_plot(df_org, orgs[[org]], major_linewidth = 0.3)
})

legend_plot <- ggplot(filter_imp, aes(x = Category, y = Term,
                                      color = Enrichment, size = Gene_frequency)) +
  geom_point() +
  theme_minimal(base_size = 9) +                         
  scale_color_viridis_c(name = "-log10 (pvalue)") +
  scale_size(name = "Gene frequency", range = c(2.0, 5.0)) +
  theme(
    legend.position       = "right",
    legend.direction      = "vertical",
    legend.title          = element_text(size = 10, face = "bold"),
    legend.text           = element_text(size = 9),
    legend.background     = element_rect(fill = "white", colour = NA),  # <- fondo blanco
    legend.box.background = element_rect(fill = "white", colour = NA),
    legend.key            = element_rect(fill = "white", colour = NA)
  )
legend <- cowplot::get_legend(legend_plot)

lots <- Map(function(p, i){
  if (i %in% 1:3) {
    p + theme(plot.margin = margin(10, 3, 5, 3))   # top, right, bottom, left
  } else if (i %in% 4:6) {
    p + theme(plot.margin = margin(5, 3, 5, 3))
  } else {
    p + theme(plot.margin = margin(5, 3, 5, 3))
  }
}, plots, seq_along(plots))


# --- Columnas (sin cambios de estilo en tus plots) ---
col1 <- cowplot::plot_grid(plots[[1]], plots[[2]], plots[[3]],
                           ncol = 1, align = "v", axis = "l")
col2 <- cowplot::plot_grid(plots[[4]], plots[[5]], plots[[6]],
                           ncol = 1, align = "v", axis = "l")
col3 <- cowplot::plot_grid(plots[[7]], plots[[8]],
    cowplot::ggdraw() +
    cowplot::draw_grob(legend,
                       x = 0.1, y = 0.5,           # centro del espacio
                       width = 0.9, height = 0.9, # padding interno (10%)
                       hjust = 0.5, vjust = 0.5), # ancla centrada
  ncol = 1, align = "v", axis = "l",
  rel_heights = c(1, 1, 1)
)

# --- Figura combinada (mantengo tu alineación vertical) ---
combined <- cowplot::plot_grid(
  col1, col2, col3,
  ncol = 3, align = "v", axis = "l",
  greedy = TRUE
)

ggsave("./Figures/Fig06_dotplot2.tiff",
       plot = combined,
       device = "tiff",
       width = 14, height = 10, units = "in",
       dpi = 300,
       bg = "white",
       compression = "lzw")

In [22]:
library(ggplot2)
library(dplyr)
library(stringr)
library(cowplot)
library(viridis)
library(ragg)

#two-column width
fig_width_mm  <- 180
fig_height_mm <- 240
dpi_out <- 300

short_term <- function(x, width = 36) {
  stringr::str_trunc(x, width = width, side = "right", ellipsis = "...")
}

make_plot <- function(df, title,
                      major_linewidth = 0.25,
                      wrap_width = 32,
                      jitter_w = 0.04,
                      jitter_h = 0.04,
                      alpha_range = c(0.30, 0.85),
                      size_range = c(1.2, 3.8)) {
  
  df2 <- df %>%
    dplyr::count(Category, Term, name = "overlap_n") %>%
    dplyr::right_join(df, by = c("Category", "Term"))
  
  ggplot(df2, aes(x = Category, y = Term,
                  color = Enrichment,
                  size  = Gene_frequency,
                  alpha = overlap_n)) +
    geom_point(
      position = position_jitter(
        width = jitter_w,
        height = jitter_h,
        seed = 1
      )
    ) +
    theme_minimal(base_size = 8, base_family = "sans") +
    scale_color_viridis_c(name = "-log10 (pvalue)") +
    scale_size(name = "Gene frequency", range = size_range) +
    scale_alpha(range = alpha_range, guide = "none") +
    scale_x_discrete(
      limits = cat_order,
      labels = c(
        "essential" = "essential",
        "noeffect"  = "mean effect",
        "enhance"   = "enhance"
      )
    ) +
    scale_y_discrete(
      labels = function(x) short_term(x, width = 36)
    ) +
    labs(
      title = stringr::str_wrap(title, width = wrap_width),
      x = NULL,
      y = NULL
    ) +
    theme(
      plot.background   = element_rect(fill = "white", colour = NA),
      panel.background  = element_rect(fill = "white", colour = NA),
      
      panel.grid.major  = element_line(color = "grey80", linewidth = major_linewidth),
      panel.grid.minor  = element_blank(),
      
      axis.line         = element_blank(),
      axis.ticks        = element_blank(),
      
      axis.text.x       = element_text(
        size = 8,
        vjust = 0.95,
        hjust = 1,
        colour = "black",
        angle = 45
      ),
      axis.text.y       = element_text(
        size = 7.6,
        colour = "black",
        lineheight = 0.9
      ),
      
      legend.position   = "none",
      
      plot.title        = element_text(
        hjust = 0.5,
        size = 8.5,
        face = "italic",
        margin = margin(b = 1)
      ),
      
      # margen izquierdo más grande para que NO se corten etiquetas
      plot.margin       = margin(t = 3, r = 3, b = 3, l = 15)
    ) +
    coord_cartesian(clip = "off")
}

# --- Organism keys and display titles ---
orgs <- list(
  "caulobacter_crescentus_na1000"       = "C. crescentus",
  "pseudomonas_fluorescens_fw300-n2c3"  = "P. fluorescens",
  "pseudomonas_putida_kt2440"           = "P. putida",
  "dinoroseobacter_shibae_dfl_12"       = "D. shibae",
  "pseudomonas_simiae_wcs417"           = "P. simiae",
  "pseudomonas_stutzeri_rch2"           = "P. stutzeri",
  "sinorhizobium_meliloti_1021"         = "S. meliloti",
  "echinicola_vietnamensis"             = "E. vietnamensis"
)

plots <- lapply(names(orgs), function(org) {
  df_org <- filter_imp %>% dplyr::filter(Organism == org)
  make_plot(df_org, orgs[[org]])
})

# --- Leyenda compacta abajo ---
legend_plot <- ggplot(
  filter_imp,
  aes(x = Category, y = Term,
      color = Enrichment,
      size = Gene_frequency)
) +
  geom_point() +
  theme_minimal(base_size = 8, base_family = "sans") +
  scale_color_viridis_c(name = "-log10 (pvalue)") +
  scale_size(name = "Gene frequency", range = c(1.2, 3.8)) +
  guides(
    color = guide_colorbar(
      title.position = "top",
      barwidth = unit(32, "mm"),
      barheight = unit(3, "mm")
    ),
    size = guide_legend(
      title.position = "top",
      nrow = 1
    )
  ) +
  theme(
    legend.position       = "bottom",
    legend.direction      = "horizontal",
    legend.box            = "horizontal",
    legend.title          = element_text(size = 8, face = "bold"),
    legend.text           = element_text(size = 8),
    legend.background     = element_rect(fill = "white", colour = NA),
    legend.box.background = element_rect(fill = "white", colour = NA),
    legend.key            = element_rect(fill = "white", colour = NA),
    plot.background       = element_rect(fill = "white", colour = NA),
    panel.background      = element_rect(fill = "white", colour = NA)
  )

legend <- cowplot::get_legend(legend_plot)

# --- 2 columnas x 4 filas ---
plot_body <- cowplot::plot_grid(
  plotlist = plots,
  ncol = 2,
  align = "none"  #evita cortes raros de etiquetas del lado izquierdo
)

combined <- cowplot::plot_grid(
  plot_body,
  legend,
  ncol = 1,
  rel_heights = c(1, 0.075)
)

combined_white <- cowplot::ggdraw() +
  cowplot::draw_plot(combined, x = 0, y = 0, width = 1, height = 1) +
  theme(
    plot.background = element_rect(fill = "white", colour = NA)
  )

ggsave(
  filename = "./Figures/Figure_S06.tiff",
  plot = combined_white,
  device = ragg::agg_tiff,
  width = fig_width_mm,
  height = fig_height_mm,
  units = "mm",
  dpi = dpi_out,
  bg = "white",
  compression = "lzw"
)

ggsave(
  filename = "./Figures/Figure_S06.pdf",
  plot = combined_white,
  width = fig_width_mm,
  height = fig_height_mm,
  units = "mm",
  device = cairo_pdf,
  bg = "white"
)

ggsave(
  filename = "./Figures/Figure_S06.png",
  plot = combined_white,
  device = ragg::agg_png,
  width = fig_width_mm,
  height = fig_height_mm,
  units = "mm",
  dpi = dpi_out,
  bg = "white"
 # compression = "lzw"
)